# Notebook 08 — Walk-Forward Evaluation Engine

## Purpose

This notebook creates the common evaluation process used by every model.

Walk-forward evaluation means:

1. Train only on information available in the past.
2. Forecast the next block of dates.
3. Move forward.
4. Retrain approximately every 21 trading days.
5. Repeat.

This prevents future-data leakage and makes the experiment closer to real forecasting.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Find the project root whether the notebook is launched from:
#   project/
# or:
#   project/notebooks/
CURRENT = Path.cwd().resolve()

if (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif CURRENT.name == "notebooks" and (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    possible_roots = [CURRENT, *CURRENT.parents]
    matches = [p for p in possible_roots if (p / "data").exists() and (p / "notebooks").exists()]
    if not matches:
        raise FileNotFoundError(
            "Could not find the project root. Open the sp500-forecasting-dissertation "
            "folder in VS Code, then run this notebook again."
        )
    PROJECT_ROOT = matches[0]

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURES = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR = PROJECT_ROOT / "reports" / "models"

for folder in [DATA_PROCESSED, REPORT_TABLES, REPORT_FIGURES, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: /home/claude/project/sp500-forecasting-dissertation
Python: /usr/bin/python3
Python version: 3.12.3


In [2]:
def find_crsp_panel(processed_folder: Path) -> Path:
    """Find the best available corrected CRSP parquet file."""
    preferred_names = [
        "crsp_sp500_daily_corrected_2010_2024.parquet",
        "crsp_sp500_daily_corrected.parquet",
        "crsp_daily_corrected_2010_2024.parquet",
        "crsp_daily_2010_2024.parquet",
    ]

    for name in preferred_names:
        path = processed_folder / name
        if path.exists():
            return path

    candidates = sorted(
        processed_folder.glob("*.parquet"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No parquet file was found in {processed_folder}. "
            "Run and save the corrected CRSP data notebook first."
        )

    print("No preferred filename found. Using the newest parquet file:")
    return candidates[0]


PANEL_PATH = find_crsp_panel(DATA_PROCESSED)
panel = pd.read_parquet(PANEL_PATH).copy()
panel.columns = [str(c).strip().lower() for c in panel.columns]

# Accept the possible names used in the earlier audit notebook.
return_candidates = ["ret", "combined_return", "ret_combined"]
return_source = next((c for c in return_candidates if c in panel.columns), None)

required_base = {"permno", "date"}
missing_base = required_base.difference(panel.columns)

if missing_base:
    raise ValueError(f"Missing required columns: {sorted(missing_base)}")

if return_source is None:
    raise ValueError(
        "Could not find a return column. Expected one of: "
        f"{return_candidates}. Found: {panel.columns.tolist()}"
    )

panel["date"] = pd.to_datetime(panel["date"])
panel["permno"] = pd.to_numeric(panel["permno"], errors="coerce").astype("Int64")
panel[return_source] = pd.to_numeric(panel[return_source], errors="coerce")

if "mktcap" in panel.columns:
    panel["mktcap"] = pd.to_numeric(panel["mktcap"], errors="coerce")

panel = (
    panel.dropna(subset=["permno", "date"])
         .sort_values(["date", "permno"])
         .reset_index(drop=True)
)

# IMPORTANT DECISION:
# "simple" keeps CRSP total returns, including a possible -100% delisting return.
# "log" uses log(1 + return), but an exact -100% return cannot be logged.
RETURN_MODE = "simple"

if RETURN_MODE == "simple":
    panel["model_return"] = panel[return_source]
elif RETURN_MODE == "log":
    impossible_for_log = panel[return_source] <= -1
    print("Rows not usable as log-returns:", int(impossible_for_log.sum()))
    panel["model_return"] = np.where(
        panel[return_source] > -1,
        np.log1p(panel[return_source]),
        np.nan,
    )
else:
    raise ValueError("RETURN_MODE must be 'simple' or 'log'.")

print("Loaded:", PANEL_PATH)
print("Rows:", len(panel))
print("Dates:", panel["date"].min(), "to", panel["date"].max())
print("Unique PERMNOs:", panel["permno"].nunique())
print("Return source:", return_source)
print("Return mode:", RETURN_MODE)
print("Missing modelling returns:", panel["model_return"].isna().sum())

Loaded: /home/claude/project/sp500-forecasting-dissertation/data/processed/crsp_sp500_daily_corrected_2010_2024.parquet
Rows: 1711517
Dates: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Unique PERMNOs: 740
Return source: ret
Return mode: simple
Missing modelling returns: 60


In [3]:
def latest_cross_section_before(data: pd.DataFrame, as_of_date) -> pd.DataFrame:
    """
    Return the cross-section on the latest trading date on or before as_of_date.

    We deliberately do NOT take each stock's individually latest historical row.
    Doing that could accidentally include a company that left the S&P 500 years ago.
    Using one common market date keeps only securities present in the corrected panel
    on that date.
    """
    as_of_date = pd.Timestamp(as_of_date)

    eligible_dates = data.loc[
        data["date"] <= as_of_date,
        "date"
    ]

    if eligible_dates.empty:
        raise ValueError(f"No data available on or before {as_of_date.date()}.")

    market_date = eligible_dates.max()

    cross_section = data.loc[
        data["date"] == market_date
    ].copy()

    if cross_section.empty:
        raise ValueError(f"No cross-section found for {market_date.date()}.")

    return cross_section


def select_top_n_by_lagged_market_cap(
    data: pd.DataFrame,
    as_of_date,
    n_stocks: int,
) -> list[int]:
    """
    Select stocks using market capitalisation from one common historical date.
    """
    if "mktcap" not in data.columns:
        raise ValueError("The panel needs a 'mktcap' column for top-N selection.")

    cross_section = latest_cross_section_before(data, as_of_date)
    cross_section = cross_section.dropna(subset=["mktcap"])
    cross_section = cross_section[cross_section["mktcap"] > 0]

    selected = (
        cross_section.nlargest(n_stocks, "mktcap")["permno"]
                     .astype(int)
                     .tolist()
    )

    if len(selected) < n_stocks:
        print(f"Warning: requested {n_stocks} stocks, but selected {len(selected)}.")

    return selected

## Quick mode versus full mode

The full experiment can be slow because ARMA and GARCH must be fitted repeatedly across many stocks.

Start with quick mode. Once it works, change the settings carefully.

In [4]:
# Matched to the exact 10-stock universe used by the LSTM/GRU/TFT notebooks
# (09_lstm_and_gru.ipynb, 10_tft_model.ipynb), over the same 2020-2024 test
# period. This replaces the earlier QUICK_MODE pilot (5 stocks, Q1 2020 only),
# which was never re-run at full scale and left notebook 12's model comparison
# collapsed to a tiny, non-representative overlap.

FIXED_UNIVERSE = [10107, 11850, 13407, 14542, 14593, 22111, 47896, 83443, 84788, 90319]

TRAIN_END = pd.Timestamp("2018-12-31")
VALIDATION_END = pd.Timestamp("2019-12-31")
TEST_START = pd.Timestamp("2020-01-02")
TEST_END = pd.Timestamp("2024-12-31")
RETRAIN_EVERY = 21
MAX_ARMA_ORDER = 2  # kept at 2 (not 5) so the full 5-year x 10-stock refit finishes in reasonable time

print({
    "universe": FIXED_UNIVERSE,
    "test_start": TEST_START,
    "test_end": TEST_END,
    "retrain_every": RETRAIN_EVERY,
    "max_arma_order": MAX_ARMA_ORDER,
})

{'universe': [10107, 11850, 13407, 14542, 14593, 22111, 47896, 83443, 84788, 90319], 'test_start': Timestamp('2020-01-02 00:00:00'), 'test_end': Timestamp('2024-12-31 00:00:00'), 'retrain_every': 21, 'max_arma_order': 2}


## Forecasting functions

This notebook uses three simple baselines:

- historical mean;
- ARMA selected by BIC;
- AR(1)-GARCH(1,1) with Student-t errors.

The GARCH specification here is deliberately simple. Confirm the final mean specification with the supervisor before final production runs.

In [5]:
from itertools import product
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model


def mean_forecast(train: pd.Series, horizon: int) -> np.ndarray:
    return np.repeat(train.mean(), horizon)


def fit_best_arma(train: pd.Series, max_order: int = 2):
    best_bic = np.inf
    best_model = None
    best_order = None

    for p, q in product(range(max_order + 1), range(max_order + 1)):
        if p == 0 and q == 0:
            continue
        try:
            fitted = ARIMA(train, order=(p, 0, q)).fit()
            if fitted.bic < best_bic:
                best_bic = fitted.bic
                best_model = fitted
                best_order = (p, q)
        except Exception:
            continue

    if best_model is None:
        raise RuntimeError("No ARMA model fitted successfully.")

    return best_model, best_order


def arma_forecast(train: pd.Series, horizon: int, max_order: int = 2):
    fitted, order = fit_best_arma(train, max_order=max_order)
    forecast = np.asarray(fitted.forecast(steps=horizon), dtype=float)
    return forecast, order


def ar_garch_forecast(train: pd.Series, horizon: int):
    scaled = train.dropna() * 100

    fitted = arch_model(
        scaled,
        mean="AR",
        lags=1,
        vol="GARCH",
        p=1,
        q=1,
        dist="t",
        rescale=False,
    ).fit(disp="off")

    forecast = fitted.forecast(horizon=horizon, reindex=False)

    mean_values = forecast.mean.values[-1, :] / 100
    variance_values = forecast.variance.values[-1, :] / (100 ** 2)

    return mean_values, variance_values

## The walk-forward loop

At each retraining date:

- select the top stocks using market cap known before the forecast block;
- use all past returns up to the day before the block;
- forecast the next block;
- save actual and predicted returns.

No test-period observation is used before it becomes historically available.

In [6]:
all_test_dates = (
    panel.loc[
        panel["date"].between(TEST_START, TEST_END),
        "date"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

prediction_rows = []

for block_start_index in range(0, len(all_test_dates), RETRAIN_EVERY):
    block_dates = all_test_dates[
        block_start_index:block_start_index + RETRAIN_EVERY
    ]

    if not block_dates:
        continue

    first_forecast_date = pd.Timestamp(block_dates[0])

    earlier_dates = panel.loc[
        panel["date"] < first_forecast_date,
        "date"
    ]

    if earlier_dates.empty:
        continue

    information_date = earlier_dates.max()

    universe = FIXED_UNIVERSE

    print(
        f"Block {first_forecast_date.date()} to "
        f"{pd.Timestamp(block_dates[-1]).date()} | "
        f"information date {information_date.date()} | "
        f"{len(universe)} stocks"
    )

    for permno in universe:
        stock_data = (
            panel.loc[
                panel["permno"] == permno,
                ["date", "model_return"]
            ]
            .dropna()
            .sort_values("date")
            .set_index("date")["model_return"]
        )

        train = stock_data.loc[:information_date]
        actual = stock_data.reindex(block_dates)

        if len(train) < 500:
            continue

        horizon = len(block_dates)

        forecasts = {
            "mean_baseline": mean_forecast(train, horizon),
        }

        arma_order = None
        try:
            arma_values, arma_order = arma_forecast(
                train,
                horizon,
                max_order=MAX_ARMA_ORDER,
            )
            forecasts["arma"] = arma_values
        except Exception as error:
            print(f"ARMA failed for {permno}: {error}")

        garch_variance = None
        try:
            garch_mean, garch_variance = ar_garch_forecast(train, horizon)
            forecasts["ar1_garch_t"] = garch_mean
        except Exception as error:
            print(f"GARCH failed for {permno}: {error}")

        for model_name, forecast_values in forecasts.items():
            for step, date in enumerate(block_dates):
                row = {
                    "date": pd.Timestamp(date),
                    "permno": int(permno),
                    "model": model_name,
                    "forecast": float(forecast_values[step]),
                    "actual": (
                        float(actual.loc[date])
                        if pd.notna(actual.loc[date])
                        else np.nan
                    ),
                    "information_date": information_date,
                    "arma_order": str(arma_order) if model_name == "arma" else None,
                    "forecast_variance": (
                        float(garch_variance[step])
                        if model_name == "ar1_garch_t" and garch_variance is not None
                        else np.nan
                    ),
                }
                prediction_rows.append(row)

predictions = pd.DataFrame(prediction_rows)
print("Prediction rows:", len(predictions))
predictions.head()

Block 2020-01-02 to 2020-01-31 | information date 2019-12-31 | 10 stocks


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


...

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Prediction rows: 37740


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


,date,permno,model,forecast,actual,information_date,arma_order,forecast_variance
0,2020-01-02,10107,mean_baseline,0.000851,0.018516,2019-12-31,None,NaN
1,2020-01-03,10107,mean_baseline,0.000851,-0.012452,2019-12-31,None,NaN
2,2020-01-06,10107,mean_baseline,0.000851,0.002585,2019-12-31,None,NaN
3,2020-01-07,10107,mean_baseline,0.000851,-0.009118,2019-12-31,None,NaN
4,2020-01-08,10107,mean_baseline,0.000851,0.015928,2019-12-31,None,NaN


## Evaluate the demonstration

These are descriptive metrics:

- RMSE: punishes large errors strongly;
- MAE: average absolute error;
- directional accuracy: percentage of correct up/down signs.

They do not by themselves tell us whether differences are statistically significant. That comes in Notebook 12.

In [7]:
def rmse(actual, forecast):
    return np.sqrt(np.mean((actual - forecast) ** 2))


def mae(actual, forecast):
    return np.mean(np.abs(actual - forecast))


def directional_accuracy(actual, forecast):
    return np.mean(np.sign(actual) == np.sign(forecast))


clean_predictions = predictions.dropna(subset=["actual", "forecast"]).copy()

metrics_rows = []

for model_name, group in clean_predictions.groupby("model"):
    metrics_rows.append({
        "model": model_name,
        "observations": len(group),
        "rmse": rmse(group["actual"].to_numpy(), group["forecast"].to_numpy()),
        "mae": mae(group["actual"].to_numpy(), group["forecast"].to_numpy()),
        "directional_accuracy": directional_accuracy(
            group["actual"].to_numpy(),
            group["forecast"].to_numpy(),
        ),
    })

walk_forward_metrics = pd.DataFrame(metrics_rows).sort_values("rmse")
walk_forward_metrics

,model,observations,rmse,mae,directional_accuracy
0,ar1_garch_t,12580,0.020350,0.013787,0.527424
1,arma,12580,0.020354,0.013790,0.526709
2,mean_baseline,12580,0.020355,0.013792,0.527981


In [8]:
predictions_path = DATA_PROCESSED / "walk_forward_classical_predictions.parquet"
metrics_path = REPORT_TABLES / "walk_forward_classical_metrics.csv"

predictions.to_parquet(predictions_path, index=False)
walk_forward_metrics.to_csv(metrics_path, index=False)

print("Saved:", predictions_path)
print("Saved:", metrics_path)

Saved: /home/claude/project/sp500-forecasting-dissertation/data/processed/walk_forward_classical_predictions.parquet
Saved: /home/claude/project/sp500-forecasting-dissertation/reports/tables/walk_forward_classical_metrics.csv


## What to say in the meeting

> Walk-forward evaluation is the anti-cheating mechanism of the dissertation. Each forecast block is produced using only information available before that block. The universe is also selected using lagged market capitalisation. All models will eventually pass through the same dates, targets and metrics so the comparison is fair.